In [ ]:
   from google.colab import drive
   drive.mount('/content/drive')

In [ ]:
!pip install pandas pyarrow

In [ ]:
"""
Filter dataset MedLayXPlain -> hanya menyisakan sample yang berhubungan dengan CHEST.
Modality bebas (CT, MRI, X-ray/CXR, dll) — yang penting bagian tubuhnya chest/thorax.

Dataset ini TIDAK punya kolom body-part/anatomy langsung. Kolom 'modality' cuma
menunjukkan jenis pemeriksaan (CT/MRI/dst), bukan bagian tubuh. Jadi filtering
dilakukan lewat keyword search pada kolom 'expert_caption' dan 'lay_caption',
karena caption-nya secara eksplisit menyebut bagian tubuh, mis:
  "The image is a CT scan of the chest, showing the lungs, heart..."

Cara pakai di GOOGLE COLAB:
1. Upload folder MedLayXPlain (berisi file .parquet) ke Google Drive kamu dulu
   (drive.google.com -> upload folder, atau sync dari desktop).
2. Di cell Colab PALING ATAS, jalankan ini dulu untuk mount Drive:
       from google.colab import drive
       drive.mount('/content/drive')
   Nanti akan muncul link buat login & authorize akses Drive.
3. Sesuaikan DRIVE_BASE_DIR di bawah dengan lokasi folder MedLayXPlain di Drive kamu
   (cek lewat file browser di sidebar kiri Colab, klik kanan folder -> Copy path).
4. Install dependency kalau belum ada: !pip install pandas pyarrow
5. Jalankan script ini.

PENTING soal performa: file di Google Drive yang di-mount itu diakses lewat jaringan
(FUSE), jadi baca berulang-ulang (seperti proses batch di sini) SANGAT lambat dan bisa
bikin RAM Colab penuh gara-gara caching di background, walau data yang kita proses
sendiri kecil. Makanya script ini otomatis COPY dulu file dari Drive ke disk lokal
Colab (/content/) sebelum diproses, dan output juga ditulis ke lokal dulu baru
dipindah balik ke Drive di akhir. Ini butuh ruang disk lokal ekstra sebesar file
sumbernya (mis. ~12GB untuk train) - Colab biasanya punya ~100GB disk lokal jadi aman.
"""

import gc
import shutil
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import re
import sys
from pathlib import Path

# ============ CONFIG ============
# Lokasi folder MedLayXPlain di Google Drive (sumber file asli).
DRIVE_BASE_DIR = "/content/drive/MyDrive/Thesis Journey/MedLayXPlain"

# Folder kerja LOKAL di Colab (bukan Drive) - file akan di-copy ke sini dulu
# sebelum diproses, karena jauh lebih cepat & stabil daripada baca langsung dari Drive.
LOCAL_WORK_DIR = "/content/medlayxplain_local"

# Nama file per split. Kalau nama file kamu beda (mis. "train-00000-of-00001.parquet"),
# ganti di sini.
SPLIT_FILES = {
    "train": "MedLayXPlain_train.parquet",
    "val": "MedLayXPlain_val.parquet",
    "test": "MedLayXPlain_test.parquet",
}

# Kolom teks tempat caption disimpan (dataset ini pasti punya kolom ini).
CAPTION_COLUMNS = ["expert_caption", "lay_caption"]

# Berapa baris yang diproses per batch. Karena sekarang file diproses dari disk
# LOKAL (bukan Drive langsung), ini bisa lebih besar. Kalau masih kehabisan RAM,
# turunkan (mis. jadi 100 atau 50).
BATCH_SIZE = 500

# Keyword yang menandakan CHEST (case-insensitive, whole-word match).
CHEST_KEYWORDS = [
    "chest", "thorax", "thoracic", "cxr", "lung", "lungs", "pulmonary",
    "cardiothoracic", "pleura", "pleural", "mediastinum", "mediastinal",
    "rib", "ribs", "hemithorax"
]
# ==============================================================


def build_chest_regex(keywords: list) -> re.Pattern:
    pattern = r"\b(" + "|".join(re.escape(k) for k in keywords) + r")\b"
    return re.compile(pattern, flags=re.IGNORECASE)


def filter_batch(df_batch: pd.DataFrame, chest_regex: re.Pattern, existing_caption_cols: list) -> pd.DataFrame:
    mask = pd.Series(False, index=df_batch.index)
    for col in existing_caption_cols:
        mask |= df_batch[col].astype(str).str.contains(chest_regex, na=False)
    return df_batch[mask]


def copy_to_local(drive_path: Path, local_path: Path):
    if local_path.exists() and local_path.stat().st_size == drive_path.stat().st_size:
        print(f"  [info] Sudah ada copy lokal yang cocok ukurannya, skip copy: {local_path}")
        return
    local_path.parent.mkdir(parents=True, exist_ok=True)
    size_mb = drive_path.stat().st_size / (1024 * 1024)
    print(f"  [copy] Menyalin dari Drive ke lokal ({size_mb:.0f} MB)... ini bisa makan waktu untuk file besar.")
    # copy dalam chunk supaya ada progress & tidak baca semua sekaligus ke RAM
    copied = 0
    chunk_size = 32 * 1024 * 1024  # 32MB per chunk
    with open(drive_path, "rb") as src, open(local_path, "wb") as dst:
        while True:
            chunk = src.read(chunk_size)
            if not chunk:
                break
            dst.write(chunk)
            copied += len(chunk)
            print(f"    ...{copied / (1024*1024):.0f} / {size_mb:.0f} MB", end="\r", flush=True)
    print()
    print(f"  [copy] Selesai disalin ke: {local_path}")


def process_split(split_name: str, file_name: str, chest_regex: re.Pattern):
    drive_file_path = Path(DRIVE_BASE_DIR) / file_name
    if not drive_file_path.exists():
        print(f"[skip] {split_name}: file tidak ditemukan di {drive_file_path}")
        return

    print(f"\n=== Split: {split_name} ===")

    # 1. copy file dari Drive ke disk lokal Colab dulu (jauh lebih cepat/stabil dibaca)
    local_file_path = Path(LOCAL_WORK_DIR) / file_name
    copy_to_local(drive_file_path, local_file_path)

    file_path = local_file_path
    parquet_file = pq.ParquetFile(file_path)
    schema = parquet_file.schema_arrow

    existing_caption_cols = [c for c in CAPTION_COLUMNS if c in schema.names]
    if not existing_caption_cols:
        raise ValueError(f"Tidak ada kolom caption ditemukan. Kolom yang ada: {schema.names}")

    # tulis output ke lokal dulu (jauh lebih cepat), baru dipindah ke Drive di akhir
    local_output_path = Path(LOCAL_WORK_DIR) / f"{split_name}_chest_only.parquet"
    final_output_path = Path(DRIVE_BASE_DIR) / f"{split_name}_chest_only.parquet"
    output_path = local_output_path
    writer = None
    total_rows = 0
    total_chest = 0
    modality_counts_before = {}
    modality_counts_after = {}

    try:
        # baca & proses per batch KECIL (all columns termasuk image), lalu langsung
        # tulis baris yang lolos & buang batch dari memory sebelum lanjut.
        for batch in parquet_file.iter_batches(batch_size=BATCH_SIZE):
            df_batch = batch.to_pandas(split_blocks=True, self_destruct=True)
            n_rows = len(df_batch)
            total_rows += n_rows

            if "modality" in df_batch.columns:
                for m, c in df_batch["modality"].value_counts().items():
                    modality_counts_before[m] = modality_counts_before.get(m, 0) + c

            mask = pd.Series(False, index=df_batch.index)
            for col in existing_caption_cols:
                mask |= df_batch[col].astype(str).str.contains(chest_regex, na=False)

            df_chest_batch = df_batch[mask]
            n_chest = len(df_chest_batch)
            total_chest += n_chest

            if n_chest > 0:
                if "modality" in df_chest_batch.columns:
                    for m, c in df_chest_batch["modality"].value_counts().items():
                        modality_counts_after[m] = modality_counts_after.get(m, 0) + c

                table_out = pa.Table.from_pandas(df_chest_batch, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(output_path, table_out.schema)
                writer.write_table(table_out)
                del table_out

            del df_batch, df_chest_batch, mask
            gc.collect()

            print(
                f"  ...progress: {total_rows} baris diproses, {total_chest} lolos filter chest",
                end="\r", flush=True,
            )
            sys.stdout.flush()

    finally:
        if writer is not None:
            writer.close()

    print()  # newline setelah progress
    print(f"[info] Total data awal: {total_rows}")
    if modality_counts_before:
        print(f"[info] Distribusi modality (semua data): {modality_counts_before}")
    print(f"[hasil] Total data chest : {total_chest}")
    print(f"[hasil] Data terbuang    : {total_rows - total_chest}")
    if modality_counts_after:
        print(f"[info] Distribusi modality (setelah filter chest): {modality_counts_after}")

    if total_chest == 0:
        print("[warn] Tidak ada data yang lolos filter, file output tidak dibuat.")
    else:
        print(f"  [copy] Memindahkan hasil dari lokal ke Drive...")
        shutil.copy2(local_output_path, final_output_path)
        print(f"[selesai] Disimpan ke: {final_output_path}")

    # bersihkan file lokal (source & output) untuk hemat disk lokal Colab
    try:
        if local_file_path.exists():
            local_file_path.unlink()
        if local_output_path.exists():
            local_output_path.unlink()
    except OSError:
        pass


def check_base_dir():
    base = Path(DRIVE_BASE_DIR)
    if base.exists() and base.is_dir():
        print(f"[ok] DRIVE_BASE_DIR ditemukan: {base}")
        print(f"[info] Isi folder: {[p.name for p in base.iterdir()]}")
        Path(LOCAL_WORK_DIR).mkdir(parents=True, exist_ok=True)
        return True

    print(f"[error] DRIVE_BASE_DIR TIDAK ditemukan: {base}")
    if not Path("/content/drive").exists():
        print("[fix] Google Drive sepertinya belum ter-mount di sesi Colab ini.")
        print("      Jalankan dulu di cell terpisah:")
        print("      from google.colab import drive")
        print("      drive.mount('/content/drive')")
    else:
        parent = base
        while parent != parent.parent and not parent.exists():
            parent = parent.parent
        if parent.exists():
            print(f"[debug] Folder terdekat yang ADA: {parent}")
            print(f"[debug] Isinya: {[p.name for p in parent.iterdir()]}")
        print("[fix] Cek path folder MedLayXPlain di Drive lewat file browser Colab")
        print("      (sidebar kiri -> ikon folder -> klik kanan folder -> Copy path),")
        print("      lalu update DRIVE_BASE_DIR sesuai itu.")
    return False


if __name__ == "__main__":
    if not check_base_dir():
        raise SystemExit(1)

    chest_regex = build_chest_regex(CHEST_KEYWORDS)
    for split_name, file_name in SPLIT_FILES.items():
        process_split(split_name, file_name, chest_regex)

In [ ]:
"""
Diagnostic ringan: cek struktur row group file parquet TANPA load datanya ke RAM.
Ini cuma baca metadata (footer file), jadi aman dan cepat walau file-nya 12GB.

Jalankan ini dulu sebelum lanjut filtering, supaya kita tahu strategi mana yang
paling cocok untuk file kamu.
"""

import pyarrow.parquet as pq
from pathlib import Path

# Sesuaikan path ini ke lokasi file train.parquet kamu
FILE_PATH = "/content/medlayxplain_local/MedLayXPlain_train.parquet"
# Kalau file belum di-copy ke lokal, bisa juga langsung cek dari Drive
# (cek metadata itu ringan, beda dengan baca isi datanya):
# FILE_PATH = "/content/drive/MyDrive/Thesis Journey/MedLayXPlain/MedLayXPlain_train.parquet"

pf = pq.ParquetFile(FILE_PATH)
metadata = pf.metadata

print(f"File: {FILE_PATH}")
print(f"Ukuran file di disk: {Path(FILE_PATH).stat().st_size / (1024**3):.2f} GB")
print(f"Total baris: {metadata.num_rows}")
print(f"Jumlah kolom: {metadata.num_columns}")
print(f"Jumlah row group: {metadata.num_row_groups}")
print()

total_image_compressed = 0
for i in range(metadata.num_row_groups):
    rg = metadata.row_group(i)
    rg_size_mb = rg.total_byte_size / (1024**2)
    print(f"Row group {i}: {rg.num_rows} baris, ~{rg_size_mb:.1f} MB (semua kolom)")
    # cek ukuran kolom 'image' spesifik kalau ada
    for c in range(rg.num_columns):
        col = rg.column(c)
        if col.path_in_schema == "image":
            img_mb = col.total_compressed_size / (1024**2)
            print(f"    -> kolom 'image' di row group ini: ~{img_mb:.1f} MB (compressed)")
            total_image_compressed += col.total_compressed_size

print()
print(f"Rata-rata baris per row group: {metadata.num_rows / metadata.num_row_groups:.0f}")
print(f"Total ukuran kolom 'image' (compressed) di semua row group: {total_image_compressed / (1024**3):.2f} GB")
print()
if metadata.num_row_groups <= 3:
    print("[kesimpulan] Row group SANGAT SEDIKIT relatif ke ukuran file -> ini kemungkinan")
    print("             besar penyebab RAM penuh. Perlu strategi beda (lihat rekomendasi).")
else:
    print("[kesimpulan] Row group cukup banyak/kecil -> pendekatan batch per row group")
    print("             seharusnya bisa jalan dengan RAM terbatas.")

In [ ]:
"""
TAHAP 1 dari 2: Identifikasi baris mana yang CHEST, TANPA menyentuh kolom 'image'.

Kenapa dipisah: dataset kamu ternyata cuma punya 1 row group untuk seluruh file,
dan kolom 'image' sendirian ~11.8 GB (untuk train). Membaca kolom 'image' SAMA
SEKALI SAJA akan memaksa PyArrow membongkar hampir seluruh isinya ke RAM, yang
pasti melebihi RAM Colab gratis (12.7 GB). Kolom teks (caption dll) jauh lebih
kecil, jadi tahap ini AMAN dijalankan sekarang di Colab gratis.

Output tahap ini: file metadata (TANPA kolom image) untuk SEMUA baris, plus
kolom baru 'is_chest' (True/False). Dari sini kamu sudah dapat:
- Jumlah pasti data chest per split
- Daftar id/index mana saja yang chest (buat verifikasi/EDA)

Tahap 2 (script terpisah) baru akan ambil gambarnya, dan itu butuh RAM lebih
besar dari yang tersedia di Colab gratis - lihat catatan di akhir output.
"""

import re
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow as pa
from pathlib import Path

DRIVE_BASE_DIR = "/content/drive/MyDrive/Thesis Journey/MedLayXPlain"
LOCAL_WORK_DIR = "/content/medlayxplain_local"

SPLIT_FILES = {
    "train": "MedLayXPlain_train.parquet",
    "val": "MedLayXPlain_val.parquet",
    "test": "MedLayXPlain_test.parquet",
}

CAPTION_COLUMNS = ["expert_caption", "lay_caption"]
CHEST_KEYWORDS = [
    "chest", "thorax", "thoracic", "cxr", "lung", "lungs", "pulmonary",
    "cardiothoracic", "pleura", "pleural", "mediastinum", "mediastinal",
    "rib", "ribs", "hemithorax"
]
CHEST_REGEX = "(?i)\\b(" + "|".join(re.escape(k) for k in CHEST_KEYWORDS) + r")\b"


def process_split(split_name: str, file_name: str):
    file_path = Path(LOCAL_WORK_DIR) / file_name
    if not file_path.exists():
        # kalau belum ada copy lokal, coba baca metadata dulu buat cek nama file di Drive
        drive_path = Path(DRIVE_BASE_DIR) / file_name
        if not drive_path.exists():
            print(f"[skip] {split_name}: file tidak ditemukan di Drive maupun lokal")
            return
        print(f"[info] {split_name}: belum ada copy lokal, membaca langsung dari Drive")
        print("       (agak lambat, tapi tahap ini ringan jadi seharusnya masih oke)")
        file_path = drive_path

    print(f"\n=== Split: {split_name} ===")
    pf = pq.ParquetFile(file_path)
    all_columns = pf.schema_arrow.names
    light_columns = [c for c in all_columns if c != "image"]
    existing_caption_cols = [c for c in CAPTION_COLUMNS if c in light_columns]

    if not existing_caption_cols:
        raise ValueError(f"Tidak ada kolom caption ditemukan. Kolom: {light_columns}")

    # baca SEMUA kolom kecuali 'image' - row group tunggal tapi tanpa image ini ringan
    table = pf.read(columns=light_columns)

    mask = pa.array([False] * table.num_rows)
    for col in existing_caption_cols:
        col_mask = pc.match_substring_regex(table[col], CHEST_REGEX)
        mask = pc.or_(mask, col_mask)

    table_with_flag = table.append_column("is_chest", mask)

    n_total = table.num_rows
    n_chest = pc.sum(mask.cast("int64")).as_py()

    print(f"[info] Total baris: {n_total}")
    print(f"[hasil] Baris chest: {n_chest}")
    print(f"[hasil] Baris non-chest: {n_total - n_chest}")

    if "modality" in light_columns:
        chest_table = table_with_flag.filter(mask)
        modality_col = chest_table["modality"]
        modality_counts = {}
        for v in modality_col.to_pylist():
            modality_counts[v] = modality_counts.get(v, 0) + 1
        print(f"[info] Distribusi modality (chest saja): {modality_counts}")

    # simpan metadata lengkap (tanpa image) + flag is_chest -> untuk EDA/verifikasi
    out_path = Path(LOCAL_WORK_DIR) / f"{split_name}_metadata_with_flag.parquet"
    Path(LOCAL_WORK_DIR).mkdir(parents=True, exist_ok=True)
    pq.write_table(table_with_flag, out_path)
    print(f"[selesai] Metadata (tanpa image) + flag chest disimpan ke: {out_path}")

    # simpan juga daftar id yang chest saja (kalau ada kolom 'id')
    if "id" in light_columns:
        chest_ids_table = table_with_flag.filter(mask).select(["id"])
        ids_path = Path(LOCAL_WORK_DIR) / f"{split_name}_chest_ids.parquet"
        pq.write_table(chest_ids_table, ids_path)
        print(f"[selesai] Daftar id chest disimpan ke: {ids_path}")


if __name__ == "__main__":
    for split_name, file_name in SPLIT_FILES.items():
        process_split(split_name, file_name)

    print("\n" + "=" * 60)
    print("TAHAP 1 SELESAI. Kamu sekarang tahu persis jumlah & id data chest.")
    print("Untuk tahap 2 (ambil gambarnya), file 'image' terlalu besar untuk")
    print("diproses di Colab gratis (RAM 12.7GB) karena cuma ada 1 row group.")
    print("Lihat rekomendasi lanjutan dari asisten untuk opsi tahap 2.")
    print("=" * 60)

In [ ]:
"""
Lihat contoh caption dari baris ber-modality Pathology / Dermoscopy yang lolos
filter chest, untuk cek apakah itu memang relevan atau false positive.

Ini baca file 'train_metadata_with_flag.parquet' yang sudah dihasilkan oleh
phase1_identify_chest.py - TIDAK perlu proses ulang dataset, jadi ringan & cepat.
"""

import pandas as pd
from pathlib import Path

LOCAL_WORK_DIR = "/content/medlayxplain_local"
SPLIT_TO_CHECK = "train"  # ganti ke "val" atau "test" kalau mau cek split lain

# modality yang mau dicek (bukan modality radiologi)
MODALITIES_TO_INSPECT = ["Pathology", "Dermoscopy"]

# berapa banyak contoh yang ditampilkan per modality
N_SAMPLES = 10


def main():
    path = Path(LOCAL_WORK_DIR) / f"{SPLIT_TO_CHECK}_metadata_with_flag.parquet"
    if not path.exists():
        print(f"[error] File tidak ditemukan: {path}")
        print("        Pastikan sudah menjalankan phase1_identify_chest.py dulu.")
        return

    df = pd.read_parquet(path)
    chest_df = df[df["is_chest"] == True]

    for modality in MODALITIES_TO_INSPECT:
        subset = chest_df[chest_df["modality"] == modality]
        print(f"\n{'='*70}")
        print(f"Modality: {modality} | Total baris ber-flag chest: {len(subset)}")
        print(f"{'='*70}")

        if len(subset) == 0:
            print("  (tidak ada)")
            continue

        sample = subset.sample(n=min(N_SAMPLES, len(subset)), random_state=42)
        for i, row in sample.iterrows():
            print(f"\n--- id: {row.get('id', i)} ---")
            print(f"expert_caption : {row['expert_caption'][:300]}")
            print(f"lay_caption    : {row['lay_caption'][:300]}")


if __name__ == "__main__":
    main()

In [ ]:
"""
REFINEMENT filter chest: kolom 'modality' di dataset ini ternyata tidak bisa
diandalkan (banyak CT/MRI chest asli yang malah ke-label 'Pathology'). Jadi
filter di sini pakai ISI CAPTION untuk menentukan radiologi vs bukan, bukan
kolom modality.

Logika:
1. Caption harus menyebut bagian tubuh chest/thorax (CHEST_KEYWORDS, sama seperti
   sebelumnya).
2. DAN caption harus menyebut istilah pencitraan radiologi (IMAGING_KEYWORDS):
   CT, MRI, X-ray, ultrasound, PET, dst.
3. DAN TIDAK menyebut istilah yang menandakan itu bukan radiologi (EXCLUDE_KEYWORDS):
   microscopic, histolog, biopsy, specimen, dermoscop, clinical photograph,
   cutaneous, erythematous, dst.

Script ini baca file '<split>_metadata_with_flag.parquet' yang SUDAH dihasilkan
oleh phase1_identify_chest.py - tidak perlu scan ulang dataset asli, jadi cepat.

Output: kolom baru 'is_chest_radiology' (hasil filter yang lebih ketat), dan
perbandingan jumlah dengan 'is_chest' (versi lama, cuma keyword chest doang).
"""

import re
import pandas as pd
from pathlib import Path

LOCAL_WORK_DIR = "/content/medlayxplain_local"
SPLITS = ["train", "val", "test"]

CAPTION_COLUMNS = ["expert_caption", "lay_caption"]

CHEST_KEYWORDS = [
    "chest", "thorax", "thoracic", "cxr", "lung", "lungs", "pulmonary",
    "cardiothoracic", "pleura", "pleural", "mediastinum", "mediastinal",
    "rib", "ribs", "hemithorax"
]

# istilah yang menandakan ini memang gambar pencitraan RADIOLOGI
IMAGING_KEYWORDS = [
    "ct scan", "computed tomography", "\\bct\\b", "mri", "magnetic resonance",
    "x-ray", "xray", "radiograph", "ultrasound", "sonograph", "\\bpet\\b",
    "\\bpet-ct\\b", "fluoroscop", "angiograph",
]

# istilah yang menandakan ini BUKAN radiologi (histologi/foto klinis/dst) ->
# kalau ada salah satu ini, baris di-exclude walau menyebut kata chest & imaging
EXCLUDE_KEYWORDS = [
    "microscop", "histolog", "biopsy", "specimen", "dermoscop",
    "clinical photograph", "cutaneous", "erythematous", "pathology slide",
    "high-power", "macroscopic clinical",
]


def build_regex(keywords: list) -> str:
    return "(?i)(" + "|".join(keywords if any(k.startswith("\\") for k in keywords) else [re.escape(k) for k in keywords]) + ")"


CHEST_REGEX = "(?i)\\b(?:" + "|".join(re.escape(k) for k in CHEST_KEYWORDS) + r")\b"
IMAGING_REGEX = "(?i)(?:" + "|".join(IMAGING_KEYWORDS) + ")"
EXCLUDE_REGEX = "(?i)(?:" + "|".join(EXCLUDE_KEYWORDS) + ")"


def process_split(split_name: str):
    path = Path(LOCAL_WORK_DIR) / f"{split_name}_metadata_with_flag.parquet"
    if not path.exists():
        print(f"[skip] {split_name}: {path} tidak ditemukan (jalankan phase1_identify_chest.py dulu)")
        return

    df = pd.read_parquet(path)
    existing_caption_cols = [c for c in CAPTION_COLUMNS if c in df.columns]

    combined_text = df[existing_caption_cols[0]].astype(str)
    for col in existing_caption_cols[1:]:
        combined_text = combined_text + " " + df[col].astype(str)

    has_chest = combined_text.str.contains(CHEST_REGEX, regex=True, na=False, flags=0)
    has_imaging = combined_text.str.contains(IMAGING_REGEX, regex=True, na=False, flags=0)
    has_exclude = combined_text.str.contains(EXCLUDE_REGEX, regex=True, na=False, flags=0)

    df["is_chest_radiology"] = has_chest & has_imaging & (~has_exclude)

    n_total = len(df)
    n_loose = int(df["is_chest"].sum()) if "is_chest" in df.columns else int(has_chest.sum())
    n_strict = int(df["is_chest_radiology"].sum())
    n_excluded_by_new_rule = int((has_chest & has_exclude).sum())

    print(f"\n=== Split: {split_name} ===")
    print(f"[info] Total baris                        : {n_total}")
    print(f"[info] is_chest (lama, cuma keyword chest) : {n_loose}")
    print(f"[info] is_chest_radiology (baru, ketat)    : {n_strict}")
    print(f"[info] Baris chest yang di-exclude karena non-radiologi: {n_excluded_by_new_rule}")

    if "modality" in df.columns:
        strict_modality = df.loc[df["is_chest_radiology"], "modality"].value_counts().to_dict()
        print(f"[info] Distribusi modality (setelah filter ketat): {strict_modality}")

    out_path = Path(LOCAL_WORK_DIR) / f"{split_name}_metadata_with_flag_v2.parquet"
    df.to_parquet(out_path, index=False)
    print(f"[selesai] Disimpan ke: {out_path}")

    if "id" in df.columns:
        ids_path = Path(LOCAL_WORK_DIR) / f"{split_name}_chest_radiology_ids.parquet"
        df.loc[df["is_chest_radiology"], ["id"]].to_parquet(ids_path, index=False)
        print(f"[selesai] Daftar id (versi ketat) disimpan ke: {ids_path}")


if __name__ == "__main__":
    for split in SPLITS:
        process_split(split)

In [ ]:
"""
Karena thesis TIDAK butuh kolom 'image' (full text saja), file
'<split>_metadata_with_flag_v2.parquet' yang sudah dihasilkan phase1b
SUDAH CUKUP - tidak perlu Tahap 2 (yang butuh RAM besar) sama sekali.

Script ini cuma langkah kosmetik terakhir: ambil baris yang is_chest_radiology
True, buang kolom flag yang sudah tidak diperlukan, dan simpan sebagai file
final ke Google Drive (ukurannya kecil karena tanpa image, jadi aman ditulis
langsung ke Drive tanpa perlu trik copy-ke-lokal lagi).
"""

import pandas as pd
from pathlib import Path

LOCAL_WORK_DIR = "/content/medlayxplain_local"
DRIVE_BASE_DIR = "/content/drive/MyDrive/Thesis Journey/MedLayXPlain"
SPLITS = ["train", "val", "test"]


def process_split(split_name: str):
    path = Path(LOCAL_WORK_DIR) / f"{split_name}_metadata_with_flag_v2.parquet"
    if not path.exists():
        print(f"[skip] {split_name}: {path} tidak ditemukan")
        return

    df = pd.read_parquet(path)
    n_before = len(df)

    final_df = df[df["is_chest_radiology"] == True].drop(
        columns=["is_chest", "is_chest_radiology"], errors="ignore"
    ).reset_index(drop=True)

    out_path = Path(DRIVE_BASE_DIR) / f"{split_name}_chest_text_final.parquet"
    final_df.to_parquet(out_path, index=False)

    print(f"[{split_name}] {n_before} baris -> {len(final_df)} baris chest (radiologi)")
    print(f"           disimpan ke: {out_path}")


if __name__ == "__main__":
    for split in SPLITS:
        process_split(split)

In [ ]:
import pandas as pd
df = pd.read_parquet("/content/drive/MyDrive/Thesis Journey/SOURCE_CODE/data/process_data/train_chest_text_final.parquet")
df[["expert_caption", "lay_caption"]].to_csv("train_chest_text.csv", index=False)
print(len(df), "rows ->", "train_chest_text.csv")
# then: from google.colab import files; files.download("train_chest_text.csv")

In [ ]:
# 1) Make sure the HuggingFace "datasets" library is up to date.
#    Colab usually has it already; this just upgrades it. "-q" = quiet output.
!pip install -q -U datasets

from datasets import load_dataset

# 2) Download MedLayBench-V from HuggingFace.
#    It's public (CC-BY-4.0), so no login is needed.
#    You may see a warning about "HF_TOKEN" missing. That's harmless, ignore it.
ds = load_dataset("hanjang/MedLayBench-V")

# 3) Drop the image column. We only need text + CUIs for the vocabulary work,
#    and leaving the images out keeps the dataset small in memory.
text = ds.remove_columns(["image"])

# 4) Sanity checks: splits, row counts, column names, one example row.
print(text)                                   # expect train 60k / validation 9.9k / test 9.93k
print(text["train"].column_names)             # expect: cuis, expert_caption, layman_caption
print(text["train"][0]["cuis"])               # a list of UMLS CUIs, e.g. ['C0040405']
print(text["train"][0]["expert_caption"][:150])

In [ ]:
import os

save_dir = "/content/drive/MyDrive/Thesis Journey/SOURCE_CODE/MedLayBench-V"
os.makedirs(save_dir, exist_ok=True)

for split in text:
    text[split].to_parquet(f"{save_dir}/{split}_text.parquet")
    print("saved", split, "-", len(text[split]), "rows")

In [ ]:
import os, re
import pandas as pd
from collections import Counter

# Where you saved the three parquet files last step
BASE   = "/content/drive/MyDrive/Thesis Journey/SOURCE_CODE/MedLayBench-V"
SPLITS = ["train", "validation", "test"]

# Read each split into a pandas table (a "DataFrame")
dfs = {s: pd.read_parquet(f"{BASE}/{s}_text.parquet") for s in SPLITS}
for s, d in dfs.items():
    print(s, d.shape, list(d.columns))   # shape = (rows, columns)

In [ ]:
def n_words(t):
    """Count words in a caption (None/NaN counts as 0)."""
    return len(str(t).split()) if isinstance(t, str) else 0

def n_cuis(x):
    """cuis is a list per row; count how many concepts it has."""
    return len(x) if x is not None else 0

for s, d in dfs.items():
    print(f"\n================ {s.upper()}  ({len(d):,} rows) ================")

    # 1) Missing / empty values: rows with no text are useless
    print("missing values per column:\n" + d.isna().sum().to_string())
    print("empty expert captions:", (d["expert_caption"].fillna("").str.strip() == "").sum())
    print("empty lay captions   :", (d["layman_caption"].fillna("").str.strip() == "").sum())

    # 2) Duplicates: the same expert caption appearing more than once
    print("duplicate expert captions:", d["expert_caption"].duplicated().sum())

    # 3) Caption length in words: shows how wordy each register is
    ew = d["expert_caption"].map(n_words)
    lw = d["layman_caption"].map(n_words)
    print(f"expert words  -> mean {ew.mean():.1f} | median {ew.median():.0f} | max {ew.max()}")
    print(f"lay words     -> mean {lw.mean():.1f} | median {lw.median():.0f} | max {lw.max()}")

    # 4) CUIs per row: how much UMLS grounding each pair carries
    nc = d["cuis"].map(n_cuis)
    print(f"CUIs per row  -> mean {nc.mean():.2f} | median {nc.median():.0f} | max {nc.max()} | rows with 0 CUIs: {(nc == 0).sum()}")

# 5) Most common CUIs and words in TRAIN: a quick look at what the data is about
train = dfs["train"]
cui_counts = Counter(c for row in train["cuis"] if row is not None for c in row)
print("\nTop 15 CUIs (train):", cui_counts.most_common(15))

STOP = {"the","of","and","in","a","is","with","to","on","are","an","this","there","at",
        "by","for","from","as","be","which","that","or","no","shows","showing","image"}
word_counts = Counter(w for t in train["expert_caption"].dropna()
                        for w in re.findall(r"[a-z]+", t.lower()) if w not in STOP and len(w) > 2)
print("\nTop 30 expert-caption words (train):", word_counts.most_common(30))

# 6) Leakage check: identical expert captions in both train and test
#    would inflate evaluation scores, so it's worth knowing about
leak = set(dfs["train"]["expert_caption"]) & set(dfs["test"]["expert_caption"])
print(f"\nExpert captions shared by train AND test: {len(leak)}")

# 7) Eyeball 3 random example pairs
for _, r in train.sample(3, random_state=0).iterrows():
    print("\nEXPERT:", r["expert_caption"][:200])
    print("LAY   :", r["layman_caption"][:200])
    print("CUIS  :", list(r["cuis"])[:8])

In [ ]:
EN_STOP = {"the","of","and","in","is","a","with","to","on","are","there","this",
           "an","as","at","by","for","from","be","which","that","or","no","shows"}

def is_english(t):
    """Heuristic English check: mostly ASCII letters + some English function words."""
    if not isinstance(t, str) or not t.strip():
        return False                                        # empty -> drop
    ascii_ratio = sum(c.isascii() for c in t) / len(t)      # non-English scripts fail this
    toks = re.findall(r"[a-z]+", t.lower())
    stop_ratio = sum(w in EN_STOP for w in toks) / max(len(toks), 1)
    short = len(toks) < 6                                   # tiny captions often have no "the/of"
    return ascii_ratio >= 0.95 and (stop_ratio >= 0.05 or short)

en = {}
for s, d in dfs.items():
    # Keep a row only if BOTH captions pass the English check
    mask = d["expert_caption"].map(is_english) & d["layman_caption"].map(is_english)
    en[s] = d[mask].reset_index(drop=True)
    print(f"{s:10s}: kept {mask.sum():,} / {len(d):,}  (dropped {(~mask).sum():,})")

    # Show a few dropped rows so you can confirm they really aren't English
    for t in d.loc[~mask, "expert_caption"].head(3):
        print("   dropped ->", str(t)[:120])

In [ ]:
# Words that make a caption chest-relevant. Matched as whole words, with an
# optional plural "s" ("rib" also matches "ribs"). Multi-word terms work too.
CHEST_TERMS = [
    # anatomy
    "chest","thorax","thoracic","lung","pulmonary","pleura","pleural","mediastinum",
    "mediastinal","heart","cardiac","pericardium","pericardial","rib","sternum","clavicle",
    "diaphragm","trachea","bronchus","bronchi","bronchial","hilum","hilar","lobe",
    "costophrenic","thoracic aorta","aortic arch","thoracic spine","lung parenchyma",
    # pathology / findings typical of chest imaging
    "pneumonia","pneumothorax","effusion","atelectasis","consolidation","cardiomegaly",
    "emphysema","pulmonary embolism","pulmonary edema","pulmonary nodule","tuberculosis",
    "bronchiectasis","pneumomediastinum","hydropneumothorax","empyema","interstitial",
    "ground glass","ground-glass","pulmonary fibrosis",
    # modality names that imply chest
    "cxr","chest x-ray","chest radiograph","chest ct",
]

# Captions that are NOT chest radiology even if they mention a chest word
EXCLUDE_TERMS = [
    "histology","histopathology","histological","h&e","hematoxylin","microscopy",
    "micrograph","immunohistochemistry","dermoscopy","dermoscopic","fundus",
    "fundoscopy","endoscopy","endoscopic","colonoscopy","mammography","mammogram",
]

def make_regex(terms):
    # Longest terms first so "chest x-ray" is tried before "chest"
    alts = "|".join(re.escape(t) for t in sorted(terms, key=len, reverse=True))
    return re.compile(r"\b(" + alts + r")s?\b", re.IGNORECASE)

CHEST_RE   = make_regex(CHEST_TERMS)
EXCLUDE_RE = make_regex(EXCLUDE_TERMS)

chest = {}
for s, d in en.items():
    # Search expert + lay captions together, same as your MedLayXPlain filter
    text_all = d["expert_caption"].fillna("") + " " + d["layman_caption"].fillna("")
    has_chest   = text_all.str.contains(CHEST_RE)
    has_exclude = text_all.str.contains(EXCLUDE_RE)
    mask = has_chest & ~has_exclude
    chest[s] = d[mask].reset_index(drop=True)
    print(f"{s:10s}: chest rows {mask.sum():,} / {len(d):,}  ({100*mask.mean():.1f}%)"
          f"  | excluded by EXCLUDE list: {(has_chest & has_exclude).sum():,}")

# Which chest words did the matching? Lets you spot a term pulling in junk.
hits = Counter(m.lower() for t in chest["train"]["expert_caption"].dropna()
                         for m in CHEST_RE.findall(t))
print("\nTop matched chest terms (train):", hits.most_common(20))

# Eyeball 5 kept rows and 5 dropped rows from train
print("\n--- KEPT (sample) ---")
for t in chest["train"]["expert_caption"].sample(5, random_state=1):
    print(" +", t[:150])
dropped = en["train"][~en["train"]["expert_caption"].isin(chest["train"]["expert_caption"])]
print("\n--- DROPPED (sample) ---")
for t in dropped["expert_caption"].sample(5, random_state=1):
    print(" -", t[:150])

In [ ]:
OUT = f"{BASE}/chest"
os.makedirs(OUT, exist_ok=True)

for s, d in chest.items():
    # Rename the lay column to match MedLayXPlain ("lay_caption"),
    # so both datasets can be merged in the next step without errors
    d = d.rename(columns={"layman_caption": "lay_caption"})
    d.to_parquet(f"{OUT}/{s}_chest_text_final.parquet")     # same naming as your MedLayXPlain files
    print("saved", s, len(d), "rows ->", f"{OUT}/{s}_chest_text_final.parquet")

In [ ]:
import os, shutil
import pandas as pd
from google.colab import drive, files


BASE    = "/content/drive/MyDrive/Thesis Journey/SOURCE_CODE/MedLayBench-V"
CSV_DIR = f"{BASE}/csv"                     # new subfolder just for the CSVs
os.makedirs(CSV_DIR, exist_ok=True)

SPLITS = ["train", "validation", "test"]

for split in SPLITS:
    # 1) Load the split you saved earlier
    df = pd.read_parquet(f"{BASE}/{split}_text.parquet")

    # 2) "cuis" holds a Python list in each row, which CSV can't store.
    #    Join it into plain text, e.g. ['C0040405','C0817096'] -> "C0040405;C0817096"
    df["cuis"] = df["cuis"].map(lambda x: ";".join(x) if x is not None else "")

    # 3) Write the CSV. index=False skips pandas' row-number column.
    #    utf-8-sig lets Excel show special characters (like °, µ) correctly.
    out = f"{CSV_DIR}/medlaybench_v_{split}.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"saved {out}  ({len(df):,} rows)")

# 4) Zip the 3 CSVs into one file and download it to your laptop
zip_path = shutil.make_archive("/content/medlaybench_v_csv", "zip", CSV_DIR)
files.download(zip_path)